In [1]:
# -------------------------------------------------------------
#  Code for: "Introduction to Integer Programming and Applications with Julia"
#  Chapter: 4 – Traveling Salesman Problem
#  Section: Exercise 2
#  Author(s): Luiz Henrique Nogueira Lorena
# -------------------------------------------------------------

using JuMP    # Modeling language
using GLPK    # Solver
using HDF5    # For reading HDF5 files
using Graphs  # For graph operations

# Load utility functions for plotting
include("utils/tsp_utils.jl")

# Function to solve TSP using lazy constraints
function solve_tsp_lazy(file_path)
    # Load the distance matrix from the HDF5 file
    D = h5read(file_path, "distance_matrix")

    # Number of locations
    n = size(D, 1)

    # Create model
    model = JuMP.Model(GLPK.Optimizer)

    # Silent mode (solver output is not printed)
    JuMP.set_silent(model)

    # Define the decision variables
    @variable(model, x[1:n, 1:n], Bin)

    # Objective function: minimize total distance
    @objective(model, Min, sum(D[i,j] * x[i,j] for i in 1:n, j in 1:n))

    # Constraints to eliminate self-loops
    @constraint(model, [i=1:n], x[i,i] == 0)

    # Each column sums to 1 (each location visited once)
    @constraint(model, [j=1:n], sum(x[i,j] for i in 1:n) == 1)

    # Each row sums to 1 (each location departs once)
    @constraint(model, [i=1:n], sum(x[i,j] for j in 1:n) == 1)

    # Define lazy constraints callback function
    function lazy_callback_function(cb_data)
        
        # Extracting the values of the decision variables
        x_val = zeros(n,n)
        for i in 1:n, j in 1:n
            x_val[i,j] = callback_value(cb_data, x[i,j])
        end

        # Build an undirected graph
        g = Graphs.SimpleGraph(n)  # From Graphs.jl
        for i in 1:n, j in 1:n
            if i != j && x_val[i, j] > 0.01
                Graphs.add_edge!(g, i, j)  # From Graphs.jl
            end
        end

        # Find connected components
        cc = Graphs.connected_components(g)
        if length(cc) > 1
            # Pick the smallest component (i.e., subtour)
            minTour = sort(cc, by=length)[1]

            # Initialize the left-hand side of the constraint
            subtourLhs = AffExpr()

            # Sum the decision variables corresponding to the subtour edges
            for i in minTour, j in minTour
                if i != j && x_val[i, j] > 0.01
                    subtourLhs += x[i, j]
                end
            end

            # Add lazy constraint to eliminate subtour
            constraint = @build_constraint(subtourLhs <= length(minTour) - 1)

            # Print the constraint for debugging
            println("Adding lazy constraint: ", constraint.func, " <= ", constraint.set.upper)

            # Submit the lazy constraint to the model
            MOI.submit(model, MOI.LazyConstraint(cb_data), constraint)
        end
    end

    # Set the lazy constraint callback function to the model
    MOI.set(model, MOI.LazyConstraintCallback(), lazy_callback_function)

    # Run the solver
    JuMP.optimize!(model)

    # Get the values of the decision variables
    x_opt = JuMP.value.(x)

    # Extract the routes from the solution
    route = [1]
    while true
        to = findfirst(x_opt[route[end], :] .> 0.5)
        to == 1 ? break : push!(route, to)
    end
    println("Route: $route")

    # Get the optimal value of the objective function
    z_opt = JuMP.objective_value(model)
    println("Optimal value: $z_opt meters")

    # Plot the solution on the map
    p = plot_tsp_solution(file_path, x_opt);
    display(p)
end

# Example usage
solve_tsp_lazy("data/tsp_exercise.h5")

Adding lazy constraint: x[4,7] + x[7,4] <= 1.0
Adding lazy constraint: x[3,5] + x[5,3] <= 1.0
Adding lazy constraint: x[1,2] + x[2,1] <= 1.0
Route: [1, 2, 5, 3, 8, 4, 7, 6]
Optimal value: 3693.2999999999997 meters


PyObject <folium.folium.Map object at 0x000002054CA27FB0>